## Análisis de Curvas de Crecimiento

In [6]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_growth_curve_validation(df, field_name="CenA01"):
    """Verifica que las correcciones de apertura sean consistentes"""
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.ravel()
    
    filters = ['F378', 'F395', 'F410', 'F430', 'F515', 'F660', 'F861']
    
    for i, filt in enumerate(filters[:7]):
        mag_2 = df[f'MAG_{filt}_2']
        mag_3 = df[f'MAG_{filt}_3']
        ap_corr = df[f'AP_CORR_{filt}_2']
        
        # Comparación magnitudes 2" vs 3"
        valid = (mag_2 < 90) & (mag_3 < 90) & (ap_corr < 90)
        diff = mag_2[valid] - mag_3[valid]
        
        axes[i].scatter(mag_2[valid], diff, alpha=0.5, s=1)
        axes[i].axhline(y=0, color='red', linestyle='--', alpha=0.7)
        axes[i].set_xlabel(f'MAG {filt} (2")')
        axes[i].set_ylabel('MAG_2" - MAG_3"')
        axes[i].set_title(f'{filt} - Aperture Consistency')
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../anac_data/Figs_val_pointSou/growth_curve_validation.png', dpi=150, bbox_inches='tight')
    plt.close()

# Cargar tus datos y ejecutar
df = pd.read_csv('../anac_data/Results/all_fields_gc_photometry_FAST_SAFE_CORRECTED_ERRORS_v17.csv')
plot_growth_curve_validation(df)

/tmp/ipykernel_6931/1006112604.py:33: DtypeWarning: Columns (108) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../anac_data/Results/all_fields_gc_photometry_FAST_SAFE_CORRECTED_ERRORS_v17.csv')


## Análisis de Relación Señal-Ruido (SNR)

In [8]:
def analyze_snr_distribution(df):
    """Analiza la distribución de SNR por filtro"""
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.ravel()
    
    filters = ['F378', 'F395', 'F410', 'F430', 'F515', 'F660', 'F861']
    
    for i, filt in enumerate(filters):
        snr_2 = df[f'SNR_{filt}_2']
        snr_3 = df[f'SNR_{filt}_3']
        mag_2 = df[f'MAG_{filt}_2']
        
        # Filtrar datos válidos
        valid = (snr_2 > 0) & (mag_2 < 90) & (snr_2 < 1000)
        
        if np.sum(valid) > 10:
            axes[i].scatter(mag_2[valid], snr_2[valid], alpha=0.5, s=1, label='2"')
            axes[i].scatter(mag_2[valid], snr_3[valid], alpha=0.5, s=1, label='3"')
            axes[i].set_xlabel(f'Magnitude {filt}')
            axes[i].set_ylabel('SNR')
            axes[i].set_title(f'SNR vs Magnitude - {filt}')
            axes[i].legend()
            axes[i].set_yscale('log')
            axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../anac_data/Figs_val_pointSou/snr_analysis.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    # Estadísticas de SNR
    print("📊 ESTADÍSTICAS DE SNR POR FILTRO (2\"):")
    for filt in filters:
        snr = df[f'SNR_{filt}_2']
        valid_snr = snr[snr > 0]
        if len(valid_snr) > 0:
            print(f"{filt}: Mediana={np.median(valid_snr):.1f}, "
                  f"Q1-Q3={np.percentile(valid_snr, 25):.1f}-{np.percentile(valid_snr, 75):.1f}")

analyze_snr_distribution(df)

📊 ESTADÍSTICAS DE SNR POR FILTRO (2"):
F378: Mediana=2.1, Q1-Q3=2.0-5.8
F395: Mediana=2.0, Q1-Q3=2.0-4.9
F410: Mediana=2.8, Q1-Q3=2.0-8.3
F430: Mediana=3.2, Q1-Q3=2.0-9.6
F515: Mediana=6.4, Q1-Q3=2.7-19.6
F660: Mediana=25.4, Q1-Q3=10.2-68.6
F861: Mediana=15.9, Q1-Q3=6.3-42.4


## Comparación con Catálogo Original de Taylor

In [12]:
def compare_with_taylor_catalog(df):
    """Compara magnitudes S-PLUS con SDSS del catálogo original"""
    
    # Mapeo aproximado de filtros S-PLUS a SDSS
    splus_to_sdss = {
        'F378': 'umag',  # u-band
        'F395': 'umag',  # u-band  
        'F410': 'gmag',  # g-band
        'F430': 'gmag',  # g-band
        'F515': 'gmag',  # g-band
        'F660': 'rmag',  # r-band
        'F861': 'zmag'   # z-band
    }
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.ravel()
    
    for i, (splus_filt, sdss_filt) in enumerate(splus_to_sdss.items()):
        if sdss_filt in df.columns and f'MAG_{splus_filt}_2' in df.columns:
            splus_mag = df[f'MAG_{splus_filt}_2']
            sdss_mag = df[sdss_filt]
            
            # Filtrar datos válidos
            valid = (splus_mag < 90) & (sdss_mag < 90) & (sdss_mag > 10) & (splus_mag > 10)
            
            if np.sum(valid) > 20:
                color_diff = splus_mag[valid] - sdss_mag[valid]
                
                axes[i].scatter(sdss_mag[valid], color_diff, alpha=0.5, s=1)
                axes[i].axhline(y=0, color='red', linestyle='--', alpha=0.7)
                axes[i].set_xlabel(f'SDSS {sdss_filt}')
                axes[i].set_ylabel(f'S-PLUS {splus_filt} - SDSS {sdss_filt}')
                axes[i].set_title(f'Color Comparison: {splus_filt} vs {sdss_filt}')
                axes[i].grid(True, alpha=0.3)
                
                # Estadísticas de la diferencia
                median_diff = np.median(color_diff)
                mad_diff = np.median(np.abs(color_diff - median_diff))
                axes[i].text(0.05, 0.95, f'Δ = {median_diff:.3f} ± {1.4826*mad_diff:.3f}', 
                           transform=axes[i].transAxes, bbox=dict(boxstyle="round", facecolor='wheat'))
    
    plt.tight_layout()
    plt.savefig('../anac_data/Figs_val_pointSou/comparison_taylor_catalog.png', dpi=150, bbox_inches='tight')
    plt.close()

compare_with_taylor_catalog(df)

## Análisis de Errores Fotométricos

In [13]:
def analyze_photometric_errors(df):
    """Analiza la consistencia de los errores fotométricos"""
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.ravel()
    
    filters = ['F378', 'F395', 'F410', 'F430', 'F515', 'F660', 'F861']
    
    for i, filt in enumerate(filters):
        mag = df[f'MAG_{filt}_2']
        magerr = df[f'MAGERR_{filt}_2']
        snr = df[f'SNR_{filt}_2']
        
        valid = (mag < 90) & (magerr < 90) & (snr > 0)
        
        if np.sum(valid) > 10:
            # Error vs Magnitud
            axes[i].scatter(mag[valid], magerr[valid], alpha=0.5, s=1)
            axes[i].set_xlabel(f'Magnitude {filt}')
            axes[i].set_ylabel('Magnitude Error')
            axes[i].set_title(f'Photometric Errors - {filt}')
            axes[i].grid(True, alpha=0.3)
            
            # Verificar relación teórica error ~ 1/SNR
            expected_error = 1.0857 / snr[valid]  # 2.5/log(10) ≈ 1.0857
            actual_error = magerr[valid]
            
            # Coeficiente de correlación
            correlation = np.corrcoef(expected_error, actual_error)[0,1]
            axes[i].text(0.05, 0.95, f'ρ = {correlation:.3f}', 
                       transform=axes[i].transAxes, bbox=dict(boxstyle="round", facecolor='wheat'))
    
    plt.tight_layout()
    plt.savefig('../anac_data/Figs_val_pointSou/error_analysis.png', dpi=150, bbox_inches='tight')
    plt.close()

analyze_photometric_errors(df)

## Métricas de Calidad Automáticas

In [11]:
def calculate_quality_metrics(df):
    """Calcula métricas automáticas de calidad"""
    
    print("🔬 MÉTRICAS DE CALIDAD DE FOTOMETRÍA S-PLUS")
    print("=" * 50)
    
    filters = ['F378', 'F395', 'F410', 'F430', 'F515', 'F660', 'F861']
    
    for filt in filters:
        print(f"\n📊 FILTRO {filt}:")
        
        # Porcentaje de detecciones exitosas
        mag_2 = df[f'MAG_{filt}_2']
        successful = np.sum(mag_2 < 90)
        total = len(mag_2)
        success_rate = (successful / total) * 100
        print(f"  • Tasa de éxito: {success_rate:.1f}% ({successful}/{total})")
        
        # SNR promedio
        snr_2 = df[f'SNR_{filt}_2']
        valid_snr = snr_2[snr_2 > 0]
        if len(valid_snr) > 0:
            print(f"  • SNR mediano: {np.median(valid_snr):.1f}")
        
        # Errores típicos
        magerr_2 = df[f'MAGERR_{filt}_2']
        valid_errors = magerr_2[magerr_2 < 90]
        if len(valid_errors) > 0:
            print(f"  • Error mediano: {np.median(valid_errors):.3f} mag")
        
        # Rango dinámico
        valid_mags = mag_2[mag_2 < 90]
        if len(valid_mags) > 0:
            print(f"  • Rango: {np.min(valid_mags):.1f} - {np.max(valid_mags):.1f} mag")
        
        # Correcciones de apertura
        ap_corr = df[f'AP_CORR_{filt}_2']
        valid_corr = ap_corr[ap_corr < 90]
        if len(valid_corr) > 0:
            print(f"  • Corrección apertura: {np.median(valid_corr):.3f} mag")

calculate_quality_metrics(df)

🔬 MÉTRICAS DE CALIDAD DE FOTOMETRÍA S-PLUS

📊 FILTRO F378:
  • Tasa de éxito: 72.8% (306949/421547)
  • SNR mediano: 2.1
  • Error mediano: 0.521 mag
  • Rango: 12.9 - 43.9 mag
  • Corrección apertura: 0.352 mag

📊 FILTRO F395:
  • Tasa de éxito: 72.1% (303759/421547)
  • SNR mediano: 2.0
  • Error mediano: 0.543 mag
  • Rango: 13.5 - 43.8 mag
  • Corrección apertura: 0.397 mag

📊 FILTRO F410:
  • Tasa de éxito: 77.6% (327217/421547)
  • SNR mediano: 2.8
  • Error mediano: 0.383 mag
  • Rango: 13.5 - 45.2 mag
  • Corrección apertura: 0.322 mag

📊 FILTRO F430:
  • Tasa de éxito: 79.1% (333652/421547)
  • SNR mediano: 3.2
  • Error mediano: 0.336 mag
  • Rango: 13.4 - 45.4 mag
  • Corrección apertura: 0.323 mag

📊 FILTRO F515:
  • Tasa de éxito: 83.7% (352735/421547)
  • SNR mediano: 6.4
  • Error mediano: 0.169 mag
  • Rango: 12.9 - 45.6 mag
  • Corrección apertura: 0.306 mag

📊 FILTRO F660:
  • Tasa de éxito: 85.4% (359962/421547)
  • SNR mediano: 25.4
  • Error mediano: 0.043 mag
  • 